In [1]:
from pathlib import Path
import sys
import sqlite3

import numpy as np
import pandas as pd

print("Python:", sys.executable)
print("Working folder:", Path.cwd())

db_path = Path("data/data.db")
print("Database exists:", db_path.is_file())

Python: /Users/jingjian/Personal/UCB_MFE/Python/UCB-MFE-python-preprogram-2026-winter/Homeworks/HW2/Jing_Jian/HW2/bin/python
Working folder: /Users/jingjian/Personal/UCB_MFE/Python/UCB-MFE-python-preprogram-2026-winter/Homeworks/HW2/Jing_Jian
Database exists: True


In [2]:
# Check that the database exists before connecting.
assert db_path.is_file(), f"Database not found: {db_path}"

with sqlite3.connect(db_path) as conn:
    tables = pd.read_sql_query(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
        """,
        conn,
    )

display(tables)

,name
0,hw2_quality_after
1,hw2_quality_before
2,hw2_quarantined
3,hw2_removed_all_prices_missing
4,market_caps
5,ohlc
6,ohlc_hw
7,ohlc_hw_cleaned


In [3]:
with sqlite3.connect(db_path) as conn:
    df_raw = pd.read_sql_query("SELECT * FROM ohlc_hw", conn)

print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]}")

display(df_raw.head())
df_raw.info()

Rows: 397,711
Columns: 8


,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


<class 'pandas.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  str    
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  str    
dtypes: float64(5), int64(1), str(2)
memory usage: 31.6 MB


In [4]:
# Summary statistics for numeric columns
display(df_raw.describe())

# Parse dates separately, preserving the original data.
parsed_dates = pd.to_datetime(df_raw["ts"], errors="coerce")

print("Earliest date:", parsed_dates.min())
print("Latest date:", parsed_dates.max())
print("Invalid or missing dates:", parsed_dates.isna().sum())

# Inspect original ticker labels before normalizing them.
print("Number of distinct ticker labels:", df_raw["ticker"].nunique())

ticker_counts = df_raw["ticker"].value_counts(dropna=False)
display(ticker_counts.rename_axis("ticker").to_frame("row_count"))

# repr() makes surrounding whitespace visible.
print("Original ticker labels:")
for label in df_raw["ticker"].drop_duplicates():
    print(repr(label))

,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


Earliest date: 1962-01-02 00:00:00
Latest date: 2024-12-06 00:00:00
Invalid or missing dates: 0
Number of distinct ticker labels: 150


,row_count
ticker,
PG,15450
MRK,15442
IBM,15433
DIS,15420
JNJ,15399
...,...
"<a target=""_blank""><span style=""color: blue"">CRM</span></a>",129
"<a target=""_blank""><span style=""color: red"">CRM</span></a>",126
"<a target=""_blank""><span style=""color: blue"">V</span></a>",106


Original ticker labels:
'CAT'
'CVX'
'HD'
'AAPL'
'TRV'
'JNJ'
'DIS'
'<a target="_blank"><span style="color: blue">CSCO</span></a>'
'SHW'
'KO'
'<a target="_blank"><span style="font-weight: bold; color: blue">HD</span></a>'
'<a target="_blank"><span style="font-weight: bold; color: blue">VZ</span></a>'
'IBM'
'PG'
'HON'
'AXP'
'MMM'
'GS'
'BA'
'CRM'
'UNH'
'NKE'
'NVDA'
'AMGN'
'CSCO'
'MCD'
'<a target="_blank"><span style="color: red">AMGN</span></a>'
'VZ'
'V'
'<a target="_blank"><span style="color: red">DIS</span></a>'
'<a target="_blank"><span style="font-weight: bold; color: blue">NKE</span></a>'
'<a target="_blank"><span style="color: blue">TRV</span></a>'
'MRK'
'WMT'
'<a target="_blank"><span style="font-weight: bold; color: red">SHW</span></a>'
'<a target="_blank"><span style="font-weight: bold; color: red">JNJ</span></a>'
'<a target="_blank"><span style="font-weight: bold; color: red">PG</span></a>'
'<a target="_blank"><span style="font-weight: bold; color: blue">JPM</span></a>'
'<a targe

In [5]:
# Work on a copy; preserve df_raw for comparison.
df = df_raw.copy()

df["ticker_original"] = df["ticker"]

df["ticker"] = (
    df["ticker"]
    .astype("string")
    .str.replace(r"<[^>]+>", "", regex=True)
    .str.strip()
    .str.upper()
)

df["ticker_normalized"] = df["ticker"].ne(df["ticker_original"])

# Convert dates and sort within each stock.
df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
df = df.sort_values(["ticker", "ts"]).reset_index(drop=True)

print("Distinct original labels:", df["ticker_original"].nunique())
print("Normalized tickers:", df["ticker"].nunique())
print("Rows with changed labels:", df["ticker_normalized"].sum())
print("Tickers:", sorted(df["ticker"].dropna().unique()))

# Show examples of the normalization.
display(
    df.loc[
        df["ticker_normalized"],
        ["ticker_original", "ticker"],
    ]
    .drop_duplicates()
    .head(10)
)

Distinct original labels: 150
Normalized tickers: 30
Rows with changed labels: 39936
Tickers: ['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'SHW', 'TRV', 'UNH', 'V', 'VZ', 'WMT']


,ticker_original,ticker
5,"<a target=""_blank""><span style=""color: red"">AA...",AAPL
10,"<a target=""_blank""><span style=""font-weight: b...",AAPL
38,"<a target=""_blank""><span style=""font-weight: b...",AAPL
235,"<a target=""_blank""><span style=""color: blue"">A...",AAPL
11956,"<a target=""_blank""><span style=""font-weight: b...",AMGN
11963,"<a target=""_blank""><span style=""color: red"">AM...",AMGN
11974,"<a target=""_blank""><span style=""color: blue"">A...",AMGN
12001,"<a target=""_blank""><span style=""font-weight: b...",AMGN
23252,"<a target=""_blank""><span style=""font-weight: b...",AMZN
23266,"<a target=""_blank""><span style=""color: blue"">A...",AMZN


In [6]:
# Original data columns, excluding our audit fields.
data_columns = list(df_raw.columns)
key_columns = ["ticker", "ts"]

# Count repeated copies beyond the first occurrence.
exact_duplicate_mask = df.duplicated(
    subset=data_columns,
    keep="first",
)

print("Extra exact duplicate rows:", exact_duplicate_mask.sum())

# Create a separate candidate dataset; keep df unchanged.
df_unique = df.loc[~exact_duplicate_mask].copy()

# After removing exact duplicates, repeated keys indicate conflicts.
conflict_mask = df_unique.duplicated(
    subset=key_columns,
    keep=False,
)

conflicting_rows = df_unique.loc[conflict_mask].sort_values(key_columns)

print("Rows involved in conflicting keys:", len(conflicting_rows))
print(
    "Number of conflicting ticker/date keys:",
    conflicting_rows[key_columns].drop_duplicates().shape[0],
)

display(conflicting_rows.head(20))

Extra exact duplicate rows: 29460
Rows involved in conflicting keys: 0
Number of conflicting ticker/date keys: 0


,ts,open,high,low,close,adj_close,volume,ticker,ticker_original,ticker_normalized


In [7]:
def detect_missing_values(data: pd.DataFrame) -> pd.DataFrame:
    """Return one issue row for each missing value in the original columns."""
    issues = []

    for column in data_columns:
        missing_rows = data.loc[data[column].isna()]

        for _, row in missing_rows.iterrows():
            issues.append({
                "ticker": row["ticker"],
                "issue_type": "missing_value",
                "issue_date": row["ts"],
                "description": f"Missing {column}",
                "severity": "medium" if column == "volume" else "high",
            })

    return pd.DataFrame(
        issues,
        columns=[
            "ticker",
            "issue_type",
            "issue_date",
            "description",
            "severity",
        ],
    )


missing_issues = detect_missing_values(df_unique)

print("Missing cells by column:")
display(df_unique[data_columns].isna().sum().to_frame("missing_count"))

print("Total missing-value issues:", len(missing_issues))
display(missing_issues.head(10))

Missing cells by column:


,missing_count
ts,0
open,0
high,0
low,0
close,0
adj_close,18374
volume,0
ticker,0


Total missing-value issues: 18374


,ticker,issue_type,issue_date,description,severity
0,AAPL,missing_value,1981-01-08,Missing adj_close,high
1,AAPL,missing_value,1981-01-13,Missing adj_close,high
2,AAPL,missing_value,1981-02-12,Missing adj_close,high
3,AAPL,missing_value,1981-04-16,Missing adj_close,high
4,AAPL,missing_value,1981-05-19,Missing adj_close,high
5,AAPL,missing_value,1981-06-04,Missing adj_close,high
6,AAPL,missing_value,1981-07-14,Missing adj_close,high
7,AAPL,missing_value,1981-07-20,Missing adj_close,high
8,AAPL,missing_value,1981-08-26,Missing adj_close,high
9,AAPL,missing_value,1981-10-23,Missing adj_close,high


In [8]:
with sqlite3.connect(db_path) as conn:
    reference_sample = pd.read_sql_query(
        "SELECT * FROM ohlc LIMIT 5",
        conn,
    )

display(reference_sample)
print("Reference columns:", reference_sample.columns.tolist())

,ts,open,high,low,close,adj_close,volume,ticker
0,2024-10-14,161.40,162.14,160.90,161.60,161.60,5964000,JNJ
1,2024-10-11,162.18,162.66,161.20,161.46,161.46,5253500,JNJ
2,2024-10-10,161.36,161.39,160.00,160.51,160.51,4793900,JNJ
3,2024-09-30,161.57,162.18,160.68,162.06,162.06,6010407,JNJ
4,2024-09-27,162.00,163.43,161.34,161.40,161.40,7553905,JNJ


Reference columns: ['ts', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'ticker']


In [9]:
with sqlite3.connect(db_path) as conn:
    reference_dates = pd.read_sql_query(
        "SELECT ticker, ts FROM ohlc",
        conn,
    )

reference_dates["ts"] = pd.to_datetime(reference_dates["ts"])
reference_dates["ticker"] = (
    reference_dates["ticker"]
    .astype("string")
    .str.replace(r"<[^>]+>", "", regex=True)
    .str.strip()
    .str.upper()
)

reference_dates = reference_dates.drop_duplicates(["ticker", "ts"])

In [10]:
def detect_missing_dates(
    data: pd.DataFrame,
    calendar: pd.DataFrame,
) -> pd.DataFrame:
    """Find absent ticker/date pairs over each ticker's reference history."""
    observed = data[["ticker", "ts"]].drop_duplicates()

    comparison = calendar[["ticker", "ts"]].drop_duplicates().merge(
        observed,
        on=["ticker", "ts"],
        how="left",
        indicator=True,
        validate="one_to_one",
    )

    missing = comparison.loc[
        comparison["_merge"].eq("left_only"),
        ["ticker", "ts"],
    ].copy()

    missing = missing.rename(columns={"ts": "issue_date"})
    missing["issue_type"] = "missing_trading_date"
    missing["description"] = "Date present in reference ohlc but absent from data"
    missing["severity"] = "high"

    return missing[
        ["ticker", "issue_type", "issue_date", "description", "severity"]
    ].sort_values(["ticker", "issue_date"]).reset_index(drop=True)


missing_date_issues = detect_missing_dates(df_unique, reference_dates)

print("Missing ticker/date pairs:", len(missing_date_issues))
display(missing_date_issues.head(10))

Missing ticker/date pairs: 0


,ticker,issue_type,issue_date,description,severity


In [11]:
# A tiny reference calendar with three expected observations.
demo_calendar = pd.DataFrame({
    "ticker": ["TEST", "TEST", "TEST"],
    "ts": pd.to_datetime(["2024-01-02", "2024-01-03", "2024-01-04"]),
})

# Deliberately remove January 3.
demo_data = demo_calendar.drop(index=1).copy()

demo_issues = detect_missing_dates(demo_data, demo_calendar)
display(demo_issues)

assert len(demo_issues) == 1
assert demo_issues.iloc[0]["issue_date"] == pd.Timestamp("2024-01-03")

print("Passed: the detector found the deliberately removed date.")

,ticker,issue_type,issue_date,description,severity
0,TEST,missing_trading_date,2024-01-03,Date present in reference ohlc but absent from...,high


Passed: the detector found the deliberately removed date.


In [12]:
gap_summary = reference_dates.groupby("ticker").agg(
    start=("ts", "min"),
    end=("ts", "max"),
)

gap_summary["missing_days"] = (
    missing_date_issues.groupby("ticker")
    .size()
    .reindex(gap_summary.index, fill_value=0)
)

for ticker, row in gap_summary.iterrows():
    print(
        f"Ticker {ticker} is missing {row['missing_days']} trading days "
        f"between {row['start']:%Y-%m-%d} and {row['end']:%Y-%m-%d}"
    )

Ticker AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18
Ticker AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06
Ticker AXP is missing 0 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CAT is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CRM is missing 0 trading days between 2004-06-23 and 2024-10-18
Ticker CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18
Ticker CVX is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker DIS is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker GS is missing 0 trading days between 1999-05-04 and 2024-10-18
Ticker HD is missing 0 trading days between 1981-09-22 and 2024-10-18
Ticker HON is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker IBM is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticke

In [13]:
def detect_price_volume_issues(data: pd.DataFrame) -> pd.DataFrame:
    """Report invalid prices, raw OHLC bounds, and volume issues."""
    issue_frames = []

    def add_issue(mask, issue_type, description, severity):
        rows = data.loc[mask, ["ticker", "ts"]].copy()
        rows = rows.rename(columns={"ts": "issue_date"})
        rows["issue_type"] = issue_type
        rows["description"] = description
        rows["severity"] = severity
        issue_frames.append(rows)

    price_columns = ["open", "high", "low", "close", "adj_close"]

    for column in price_columns:
        values = data[column]

        # Missing values are already recorded by detect_missing_values.
        invalid = values.notna() & (
            ~np.isfinite(values) | values.le(0)
        )

        add_issue(
            invalid,
            "invalid_price",
            f"{column} is nonpositive or non-finite",
            "high",
        )

    # Only compare finite, nonmissing prices.
    for column in ["open", "close"]:
        comparable = (
            np.isfinite(data[column])
            & np.isfinite(data["low"])
            & np.isfinite(data["high"])
        )

        outside_range = comparable & (
            data[column].lt(data["low"])
            | data[column].gt(data["high"])
        )

        add_issue(
            outside_range,
            "invalid_ohlc_bounds",
            f"{column} lies outside [low, high]",
            "high",
        )

    add_issue(
        data["volume"].eq(0),
        "zero_volume",
        "Recorded volume is zero; review for possible stale data",
        "low",
    )

    invalid_volume = data["volume"].notna() & (
        ~np.isfinite(data["volume"]) | data["volume"].lt(0)
    )

    add_issue(
        invalid_volume,
        "invalid_volume",
        "Volume is negative or non-finite",
        "high",
    )

    return pd.concat(issue_frames, ignore_index=True)[
        ["ticker", "issue_type", "issue_date", "description", "severity"]
    ]


price_volume_issues = detect_price_volume_issues(df_unique)

display(
    price_volume_issues.groupby(
        ["issue_type", "description", "severity"]
    ).size().reset_index(name="issue_count")
)

display(price_volume_issues.head(10))

,issue_type,description,severity,issue_count
0,invalid_ohlc_bounds,"open lies outside [low, high]",high,5
1,invalid_price,open is nonpositive or non-finite,high,5
2,zero_volume,Recorded volume is zero; review for possible s...,low,75


,ticker,issue_type,issue_date,description,severity
0,CVX,invalid_price,1962-01-02,open is nonpositive or non-finite,high
1,JNJ,invalid_price,1962-01-02,open is nonpositive or non-finite,high
2,MCD,invalid_price,1966-07-05,open is nonpositive or non-finite,high
3,MRK,invalid_price,1962-01-02,open is nonpositive or non-finite,high
4,PG,invalid_price,1962-01-02,open is nonpositive or non-finite,high
5,CVX,invalid_ohlc_bounds,1962-01-02,"open lies outside [low, high]",high
6,JNJ,invalid_ohlc_bounds,1962-01-02,"open lies outside [low, high]",high
7,MCD,invalid_ohlc_bounds,1966-07-05,"open lies outside [low, high]",high
8,MRK,invalid_ohlc_bounds,1962-01-02,"open lies outside [low, high]",high
9,PG,invalid_ohlc_bounds,1962-01-02,"open lies outside [low, high]",high


In [14]:
def detect_return_outliers(data: pd.DataFrame) -> pd.DataFrame:
    """Flag simple close returns beyond three sample standard deviations."""
    ordered = data.sort_values(["ticker", "ts"]).copy()

    previous_close = ordered.groupby("ticker")["close"].shift(1)

    # Only calculate returns from positive, finite prices.
    valid_pair = (
        np.isfinite(ordered["close"])
        & ordered["close"].gt(0)
        & np.isfinite(previous_close)
        & previous_close.gt(0)
    )

    ordered["simple_return"] = (
        ordered["close"] / previous_close - 1
    ).where(valid_pair)

    grouped_returns = ordered.groupby("ticker")["simple_return"]

    mean_return = grouped_returns.transform("mean")
    std_return = grouped_returns.transform(
        lambda returns: returns.std(ddof=1)
    )

    # Zero or undefined standard deviation produces no outlier flag.
    flag = (
        ordered["simple_return"].notna()
        & std_return.notna()
        & std_return.gt(0)
        & (ordered["simple_return"] - mean_return).abs().gt(3 * std_return)
    )

    issues = ordered.loc[flag, ["ticker", "ts"]].copy()
    issues = issues.rename(columns={"ts": "issue_date"})
    issues["issue_type"] = "extreme_return"
    issues["description"] = ordered.loc[flag, "simple_return"].map(
        lambda value: (
            f"Raw-close simple return {value:.2%} exceeds "
            "three standard deviations from ticker mean"
        )
    )
    issues["severity"] = "medium"

    return issues[
        ["ticker", "issue_type", "issue_date", "description", "severity"]
    ]


return_issues = detect_return_outliers(df_unique)

print("Extreme-return flags:", len(return_issues))
display(return_issues.head(10))

Extreme-return flags: 5056


,ticker,issue_type,issue_date,description,severity
9,AAPL,extreme_return,1980-12-26,Raw-close simple return 9.23% exceeds three st...,medium
83,AAPL,extreme_return,1981-04-02,Raw-close simple return 8.77% exceeds three st...,medium
217,AAPL,extreme_return,1981-09-25,Raw-close simple return -12.98% exceeds three ...,medium
347,AAPL,extreme_return,1982-03-19,Raw-close simple return 9.02% exceeds three st...,medium
530,AAPL,extreme_return,1982-11-22,Raw-close simple return -8.91% exceeds three s...,medium
536,AAPL,extreme_return,1982-11-30,Raw-close simple return 10.39% exceeds three s...,medium
578,AAPL,extreme_return,1983-01-20,Raw-close simple return 11.15% exceeds three s...,medium
647,AAPL,extreme_return,1983-04-20,Raw-close simple return 8.87% exceeds three st...,medium
720,AAPL,extreme_return,1983-07-26,Raw-close simple return -9.27% exceeds three s...,medium
747,AAPL,extreme_return,1983-08-31,Raw-close simple return 13.31% exceeds three s...,medium


In [15]:
quality_report_before = pd.concat(
    [
        missing_issues,
        missing_date_issues,
        price_volume_issues,
        return_issues,
    ],
    ignore_index=True,
).sort_values(
    ["ticker", "issue_date", "issue_type"]
).reset_index(drop=True)

print("Total issue entries:", len(quality_report_before))
print(
    "Distinct ticker/date pairs with issues:",
    quality_report_before[["ticker", "issue_date"]]
    .drop_duplicates()
    .shape[0],
)

display(
    quality_report_before.groupby(
        ["issue_type", "severity"]
    ).size().reset_index(name="issue_count")
)

display(quality_report_before.head(15))

Total issue entries: 23515
Distinct ticker/date pairs with issues: 23264


,issue_type,severity,issue_count
0,extreme_return,medium,5056
1,invalid_ohlc_bounds,high,5
2,invalid_price,high,5
3,missing_value,high,18374
4,zero_volume,low,75


,ticker,issue_type,issue_date,description,severity
0,AAPL,extreme_return,1980-12-26,Raw-close simple return 9.23% exceeds three st...,medium
1,AAPL,missing_value,1981-01-08,Missing adj_close,high
2,AAPL,missing_value,1981-01-13,Missing adj_close,high
3,AAPL,missing_value,1981-02-12,Missing adj_close,high
4,AAPL,extreme_return,1981-04-02,Raw-close simple return 8.77% exceeds three st...,medium
5,AAPL,missing_value,1981-04-16,Missing adj_close,high
6,AAPL,missing_value,1981-05-19,Missing adj_close,high
7,AAPL,missing_value,1981-06-04,Missing adj_close,high
8,AAPL,missing_value,1981-07-14,Missing adj_close,high
9,AAPL,missing_value,1981-07-20,Missing adj_close,high


### Data quality severity

- **High:** Missing prices or identifiers, invalid prices, impossible OHLC
  relationships, invalid volume, or missing expected observations.
  These prevent reliable use of the affected values.
- **Medium:** Extreme returns require investigation but are not automatically
  errors. Missing volume is unknown rather than evidence of no trading.
- **Low:** Recorded zero volume warrants review but does not by itself prove
  that a record is invalid.

The report preserves detected issues before repairs. Exact duplicate removals
are counted separately; original ticker labels and normalization flags are
retained for auditing.

In [16]:
with sqlite3.connect(db_path) as conn:
    reference = pd.read_sql_query(
        "SELECT ticker, ts, open, high, low, close, adj_close FROM ohlc",
        conn,
    )

reference["ts"] = pd.to_datetime(reference["ts"])
reference["ticker"] = (
    reference["ticker"]
    .astype("string")
    .str.replace(r"<[^>]+>", "", regex=True)
    .str.strip()
    .str.upper()
)

reference = reference.drop_duplicates()

assert not reference.duplicated(["ticker", "ts"]).any(), (
    "Reference contains conflicting ticker/date keys."
)

# Prefix reference prices so they remain distinct from homework prices.
reference = reference.rename(
    columns={
        column: f"ref_{column}"
        for column in ["open", "high", "low", "close", "adj_close"]
    }
)

comparison = df_unique.merge(
    reference,
    on=["ticker", "ts"],
    how="left",
    validate="one_to_one",
    indicator="reference_match",
)

print(comparison["reference_match"].value_counts())

# Investigate the invalid opening prices.
invalid_open = (
    comparison["open"].notna()
    & (
        ~np.isfinite(comparison["open"])
        | comparison["open"].le(0)
    )
)

display(
    comparison.loc[
        invalid_open,
        [
            "ticker", "ts",
            "open", "high", "low", "close",
            "ref_open", "ref_high", "ref_low", "ref_close",
        ],
    ]
)

reference_match
both          368251
left_only          0
right_only         0
Name: count, dtype: int64


,ticker,ts,open,high,low,close,ref_open,ref_high,ref_low,ref_close
87103,CVX,1962-01-02,0.0,3.30000,3.24000,3.30000,0.0,3.30000,3.24000,3.30000
167733,JNJ,1962-01-02,0.0,0.22338,0.22222,0.22338,0.0,0.22338,0.22222,0.22338
213412,MCD,1966-07-05,0.0,0.27366,0.26749,0.26955,0.0,0.27366,0.26749,0.26955
243893,MRK,1962-01-02,0.0,0.39703,0.39317,0.39537,0.0,0.39703,0.39317,0.39537
287040,PG,1962-01-02,0.0,1.44000,1.42000,1.43000,0.0,1.44000,1.42000,1.43000


In [17]:
# A usable date-specific factor requires positive, finite reference prices.
valid_reference = (
    np.isfinite(comparison["ref_close"])
    & comparison["ref_close"].gt(0)
    & np.isfinite(comparison["ref_adj_close"])
    & comparison["ref_adj_close"].gt(0)
)

comparison["reference_factor"] = (
    comparison["ref_adj_close"] / comparison["ref_close"]
).where(valid_reference)

comparison["expected_adj_close"] = (
    comparison["close"] * comparison["reference_factor"]
)

# Compare existing adjusted prices using a small numerical tolerance.
comparable = (
    valid_reference
    & np.isfinite(comparison["close"])
    & comparison["close"].gt(0)
    & np.isfinite(comparison["adj_close"])
    & comparison["adj_close"].gt(0)
)

comparison["adjustment_mismatch"] = comparable & ~np.isclose(
    comparison["adj_close"],
    comparison["expected_adj_close"],
    rtol=1e-6,
    atol=1e-8,
)

print("Rows without a usable reference factor:", (~valid_reference).sum())
print("Missing adjusted closes:", comparison["adj_close"].isna().sum())
print("Existing adjusted closes inconsistent with reference:",
      comparison["adjustment_mismatch"].sum())

display(
    comparison.loc[
        comparison["adjustment_mismatch"],
        [
            "ticker", "ts", "close", "adj_close",
            "ref_close", "ref_adj_close",
            "reference_factor", "expected_adj_close",
        ],
    ].head(10)
)

Rows without a usable reference factor: 0
Missing adjusted closes: 18374
Existing adjusted closes inconsistent with reference: 10417


,ticker,ts,close,adj_close,ref_close,ref_adj_close,reference_factor,expected_adj_close
35,AAPL,1981-02-03,0.123330,1.264587,0.123330,0.095069,0.770853,0.095069
93,AAPL,1981-04-28,0.126120,2.036653,0.126120,0.097220,0.770854,0.097220
101,AAPL,1981-05-08,0.125000,2.129290,0.125000,0.096360,0.770878,0.096360
157,AAPL,1981-07-29,0.106030,0.506329,0.106030,0.081734,0.770857,0.081734
163,AAPL,1981-08-06,0.112720,2.009828,0.112720,0.086896,0.770899,0.086896
167,AAPL,1981-08-13,0.103800,1.122224,0.103800,0.080013,0.770841,0.080013
168,AAPL,1981-08-14,0.102120,1.171347,0.102120,0.078722,0.770878,0.078722
206,AAPL,1981-10-08,0.082589,1.245215,0.082589,0.063666,0.770878,0.063666
264,AAPL,1981-12-31,0.098772,1.730921,0.098772,0.076141,0.770878,0.076141
274,AAPL,1982-01-15,0.089286,1.577125,0.089286,0.068829,0.770878,0.068829


In [18]:
adjustment_issues = comparison.loc[
    comparison["adjustment_mismatch"],
    ["ticker", "ts"],
].copy()

adjustment_issues = adjustment_issues.rename(
    columns={"ts": "issue_date"}
)

adjustment_issues["issue_type"] = "adjustment_mismatch"
adjustment_issues["description"] = (
    "Adjusted close differs from raw close multiplied by the "
    "matched ohlc date-specific adjustment factor "
    "(rtol=1e-6, atol=1e-8)"
)
adjustment_issues["severity"] = "high"

# Rebuild from the original issue sets so rerunning adds no duplicates.
quality_report_before = pd.concat(
    [
        missing_issues,
        missing_date_issues,
        price_volume_issues,
        return_issues,
        adjustment_issues,
    ],
    ignore_index=True,
).sort_values(
    ["ticker", "issue_date", "issue_type"]
).reset_index(drop=True)

display(
    quality_report_before.groupby(
        ["issue_type", "severity"]
    ).size().reset_index(name="issue_count")
)

print("Total issue entries:", len(quality_report_before))

,issue_type,severity,issue_count
0,adjustment_mismatch,high,10417
1,extreme_return,medium,5056
2,invalid_ohlc_bounds,high,5
3,invalid_price,high,5
4,missing_value,high,18374
5,zero_volume,low,75


Total issue entries: 33932


In [19]:
review = comparison.sort_values(["ticker", "ts"]).copy()

review["previous_close"] = (
    review.groupby("ticker")["close"].shift(1)
)
review["previous_ref_close"] = (
    review.groupby("ticker")["ref_close"].shift(1)
)

review["simple_return"] = (
    review["close"] / review["previous_close"] - 1
)
review["reference_return"] = (
    review["ref_close"] / review["previous_ref_close"] - 1
)

review["both_closes_match_reference"] = (
    np.isclose(
        review["close"],
        review["ref_close"],
        rtol=1e-6,
        atol=1e-8,
    )
    & np.isclose(
        review["previous_close"],
        review["previous_ref_close"],
        rtol=1e-6,
        atol=1e-8,
    )
)

outlier_review = return_issues[["ticker", "issue_date"]].merge(
    review,
    left_on=["ticker", "issue_date"],
    right_on=["ticker", "ts"],
    how="left",
    validate="one_to_one",
)

print("Flagged returns:", len(outlier_review))
print(
    "Both closes match reference:",
    outlier_review["both_closes_match_reference"].sum(),
)
print(
    "Require further investigation:",
    (~outlier_review["both_closes_match_reference"]).sum(),
)

# Show the largest absolute flagged returns.
display(
    outlier_review.assign(
        absolute_return=outlier_review["simple_return"].abs()
    )
    .sort_values("absolute_return", ascending=False)
    [
        [
            "ticker", "ts", "previous_close", "close",
            "simple_return", "reference_return",
            "both_closes_match_reference",
        ]
    ]
    .head(15)
)

Flagged returns: 5056
Both closes match reference: 5056
Require further investigation: 0


,ticker,ts,previous_close,close,simple_return,reference_return,both_closes_match_reference
4429,TRV,1981-08-10,6.03000,12.060000,1.000000,1.000000,True
4425,TRV,1976-10-28,4.61000,9.220000,1.000000,1.000000,True
4427,TRV,1978-12-20,4.02000,8.030000,0.997512,0.997512,True
3066,MCD,1968-01-02,0.70988,0.174900,-0.753620,-0.753620,True
98,AAPL,2000-09-29,0.95536,0.459820,-0.518695,-0.518695,True
4430,TRV,1981-08-11,12.06000,6.030000,-0.500000,-0.500000,True
4426,TRV,1976-10-29,9.22000,4.690000,-0.491323,-0.491323,True
4428,TRV,1978-12-21,8.03000,4.090000,-0.490660,-0.490660,True
3954,NVDA,2000-03-07,0.12188,0.173570,0.424106,0.424106,True
3996,NVDA,2004-08-06,0.12133,0.078584,-0.352314,-0.352314,True


### Cleaning decisions

- Remove 29,460 duplicate copies after normalizing ticker labels.
  Preserve the original data and ticker-normalization audit.
- Quarantine the five rows with zero opening prices. These are leading
  observations with no previous price available, and their reference
  opening prices are also zero.
- Reconstruct missing or inconsistent adjusted closes using validated raw
  close multiplied by the matched reference row's adj_close / close factor.
  Do not carry adjustment factors across dates. Preserve repair flags.
- Retain the 5,056 extreme-return observations with review flags.
  Both closes used in each return match the supplied reference, but this
  does not establish the cause or rule out shared source errors.
  No split or news-event explanation has been verified.
- Retain recorded zero-volume observations with review flags.
  Zero volume alone does not establish an error.
- Missing raw prices and missing volume were absent from this dataset.
  Demonstrate their required handling separately using injected examples.

In [20]:
extreme_return_keys = return_issues[
    ["ticker", "issue_date"]
].rename(columns={"issue_date": "ts"})

extreme_return_keys["extreme_return_flag"] = True

working = comparison.merge(
    extreme_return_keys,
    on=["ticker", "ts"],
    how="left",
    validate="one_to_one",
)

working["extreme_return_flag"] = (
    working["extreme_return_flag"].eq(True)
)
working["zero_volume_flag"] = working["volume"].eq(0)

print("Working rows:", len(working))
print("Extreme-return flags:", working["extreme_return_flag"].sum())
print("Zero-volume flags:", working["zero_volume_flag"].sum())

Working rows: 368251
Extreme-return flags: 5056
Zero-volume flags: 75


In [21]:
def clean_raw_data(data: pd.DataFrame):
    """Return raw-price candidates, removed rows, and quarantined rows."""
    result = data.sort_values(["ticker", "ts"]).copy()
    raw_columns = ["open", "high", "low", "close"]

    # Remove completely missing raw-price rows before forward-filling.
    all_missing = result[raw_columns].isna().all(axis=1)
    removed = result.loc[all_missing].copy()
    removed["removal_reason"] = "All four raw prices missing"

    result = result.loc[~all_missing].copy()

    # Record exactly which missing cells were successfully filled.
    for column in raw_columns:
        was_missing = result[column].isna()

        result[column] = result.groupby("ticker")[column].ffill()

        result[f"{column}_filled"] = (
            was_missing & result[column].notna()
        )

    result["volume_filled"] = result["volume"].isna()
    result["volume"] = result["volume"].fillna(0)

    # Recheck every retained raw price after filling.
    valid_prices = (
        np.isfinite(result[raw_columns]).all(axis=1)
        & result[raw_columns].gt(0).all(axis=1)
    )

    valid_bounds = (
        result["low"].le(result["open"])
        & result["open"].le(result["high"])
        & result["low"].le(result["close"])
        & result["close"].le(result["high"])
    )

    valid_volume = (
        np.isfinite(result["volume"])
        & result["volume"].ge(0)
    )

    result["quarantine_reason"] = ""

    for invalid, reason in [
        (~valid_prices, "Missing, non-finite, or nonpositive raw price; "),
        (~valid_bounds, "Invalid raw OHLC bounds; "),
        (~valid_volume, "Invalid volume; "),
    ]:
        result.loc[invalid, "quarantine_reason"] += reason

    quarantine_mask = result["quarantine_reason"].ne("")

    quarantined = result.loc[quarantine_mask].copy()
    candidates = result.loc[~quarantine_mask].copy()

    return candidates, removed, quarantined


raw_candidates, removed_rows, raw_quarantine = clean_raw_data(working)

print("Raw-price candidates:", len(raw_candidates))
print("Removed rows:", len(removed_rows))
print("Quarantined rows:", len(raw_quarantine))

display(
    raw_quarantine[
        ["ticker", "ts", "open", "quarantine_reason"]
    ]
)

assert len(working) == (
    len(raw_candidates) + len(removed_rows) + len(raw_quarantine)
)

Raw-price candidates: 368246
Removed rows: 0
Quarantined rows: 5


,ticker,ts,open,quarantine_reason
87103,CVX,1962-01-02,0.0,"Missing, non-finite, or nonpositive raw price;..."
167733,JNJ,1962-01-02,0.0,"Missing, non-finite, or nonpositive raw price;..."
213412,MCD,1966-07-05,0.0,"Missing, non-finite, or nonpositive raw price;..."
243893,MRK,1962-01-02,0.0,"Missing, non-finite, or nonpositive raw price;..."
287040,PG,1962-01-02,0.0,"Missing, non-finite, or nonpositive raw price;..."


In [22]:
def clean_adjusted_close(data: pd.DataFrame):
    """Repair adjusted close using matched date-specific reference factors."""
    result = data.copy()
    result["adj_close_original"] = result["adj_close"]

    valid_reference = (
        np.isfinite(result["ref_close"])
        & result["ref_close"].gt(0)
        & np.isfinite(result["ref_adj_close"])
        & result["ref_adj_close"].gt(0)
    )

    factor = (
        result["ref_adj_close"] / result["ref_close"]
    ).where(valid_reference)

    expected = result["close"] * factor

    usable_expected = (
        np.isfinite(factor)
        & factor.gt(0)
        & np.isfinite(expected)
        & expected.gt(0)
    )

    existing_valid = (
        np.isfinite(result["adj_close"])
        & result["adj_close"].gt(0)
    )

    consistent = existing_valid & np.isclose(
        result["adj_close"],
        expected,
        rtol=1e-6,
        atol=1e-8,
    )

    repair = usable_expected & ~consistent

    result["adj_close_repaired"] = repair
    result["adjustment_factor_used"] = factor
    result["adjustment_source"] = "ohlc: matched ticker and ts"

    result.loc[repair, "adj_close"] = expected.loc[repair]

    # Quarantine if the convention cannot be established or verified.
    resolved = (
        usable_expected
        & np.isfinite(result["adj_close"])
        & result["adj_close"].gt(0)
        & np.isclose(
            result["adj_close"],
            expected,
            rtol=1e-6,
            atol=1e-8,
        )
    )

    quarantined = result.loc[~resolved].copy()
    quarantined["quarantine_reason"] = (
        "Unresolved adjusted-close consistency"
    )

    cleaned = result.loc[resolved].copy()

    return cleaned, quarantined


cleaned, adjustment_quarantine = clean_adjusted_close(raw_candidates)

quarantined_rows = pd.concat(
    [raw_quarantine, adjustment_quarantine],
    ignore_index=True,
)

print("Retained rows:", len(cleaned))
print("Adjusted closes repaired:", cleaned["adj_close_repaired"].sum())
print("Additional adjustment quarantines:", len(adjustment_quarantine))
print("Total quarantined rows:", len(quarantined_rows))

display(
    cleaned.loc[
        cleaned["adj_close_repaired"],
        [
            "ticker", "ts", "close",
            "adj_close_original", "adjustment_factor_used", "adj_close",
        ],
    ].head(10)
)

Retained rows: 368246
Adjusted closes repaired: 28791
Additional adjustment quarantines: 0
Total quarantined rows: 5


,ticker,ts,close,adj_close_original,adjustment_factor_used,adj_close
17,AAPL,1981-01-08,0.13505,NaN,0.770826,0.104100
20,AAPL,1981-01-13,0.13616,NaN,0.770858,0.104960
35,AAPL,1981-02-03,0.12333,1.264587,0.770853,0.095069
42,AAPL,1981-02-12,0.11663,NaN,0.770872,0.089907
86,AAPL,1981-04-16,0.11161,NaN,0.770858,0.086035
93,AAPL,1981-04-28,0.12612,2.036653,0.770854,0.097220
101,AAPL,1981-05-08,0.12500,2.129290,0.770878,0.096360
108,AAPL,1981-05-19,0.12277,NaN,0.770866,0.094639
119,AAPL,1981-06-04,0.14341,NaN,0.770936,0.110560
146,AAPL,1981-07-14,0.10603,NaN,0.770857,0.081734


In [23]:
price_columns = ["open", "high", "low", "close", "adj_close"]

assert cleaned[price_columns].notna().all().all(), "Missing prices remain"
assert np.isfinite(cleaned[price_columns]).all().all(), "Non-finite prices remain"
assert cleaned[price_columns].gt(0).all().all(), "Nonpositive prices remain"

assert cleaned["ts"].notna().all(), "Missing dates remain"
assert cleaned["ticker"].notna().all(), "Missing tickers remain"
assert cleaned["ticker"].str.strip().ne("").all(), "Empty tickers remain"
assert not cleaned.duplicated(["ticker", "ts"]).any(), "Duplicate keys remain"

assert (
    cleaned["low"].le(cleaned["open"])
    & cleaned["open"].le(cleaned["high"])
    & cleaned["low"].le(cleaned["close"])
    & cleaned["close"].le(cleaned["high"])
).all(), "Invalid OHLC bounds remain"

assert (
    np.isfinite(cleaned["volume"])
    & cleaned["volume"].ge(0)
).all(), "Invalid volume remains"

assert np.isclose(
    cleaned["adj_close"],
    cleaned["close"] * cleaned["adjustment_factor_used"],
    rtol=1e-6,
    atol=1e-8,
).all(), "Adjusted-price inconsistency remains"

duplicate_copies_removed = int(exact_duplicate_mask.sum())

assert len(df_raw) == (
    duplicate_copies_removed
    + len(removed_rows)
    + len(quarantined_rows)
    + len(cleaned)
), "Row accounting does not balance"

print("All validation checks passed.")
print(f"Original rows:             {len(df_raw):,}")
print(f"Duplicate copies removed:  {duplicate_copies_removed:,}")
print(f"All-price-missing removed: {len(removed_rows):,}")
print(f"Quarantined:               {len(quarantined_rows):,}")
print(f"Retained:                  {len(cleaned):,}")

All validation checks passed.
Original rows:             397,711
Duplicate copies removed:  29,460
All-price-missing removed: 0
Quarantined:               5
Retained:                  368,246


In [24]:
quality_report_after = pd.concat(
    [
        detect_missing_values(cleaned),
        detect_missing_dates(cleaned, reference_dates),
        detect_price_volume_issues(cleaned),
        detect_return_outliers(cleaned),
    ],
    ignore_index=True,
).sort_values(
    ["ticker", "issue_date", "issue_type"]
).reset_index(drop=True)

quality_report_after["issue_date"] = pd.to_datetime(
    quality_report_after["issue_date"]
)
print("Issues after cleaning:")
display(
    quality_report_after.groupby(
        ["issue_type", "severity"]
    ).size().reset_index(name="issue_count")
)

# Identify missing dates that are explained by quarantine.
after_gaps = quality_report_after.loc[
    quality_report_after["issue_type"].eq("missing_trading_date")
]

display(
    after_gaps.merge(
        quarantined_rows[["ticker", "ts", "quarantine_reason"]],
        left_on=["ticker", "issue_date"],
        right_on=["ticker", "ts"],
        how="left",
        validate="one_to_one",
    )[["ticker", "issue_date", "quarantine_reason"]]
)

print("Before cleaning — numeric statistics:")
display(df_unique[data_columns].describe())

print("After cleaning — numeric statistics:")
display(cleaned[data_columns].describe())

Issues after cleaning:


,issue_type,severity,issue_count
0,extreme_return,medium,5057
1,missing_trading_date,high,5
2,zero_volume,low,74


,ticker,issue_date,quarantine_reason
0,CVX,1962-01-02,"Missing, non-finite, or nonpositive raw price;..."
1,JNJ,1962-01-02,"Missing, non-finite, or nonpositive raw price;..."
2,MCD,1966-07-05,"Missing, non-finite, or nonpositive raw price;..."
3,MRK,1962-01-02,"Missing, non-finite, or nonpositive raw price;..."
4,PG,1962-01-02,"Missing, non-finite, or nonpositive raw price;..."


Before cleaning — numeric statistics:


,ts,open,high,low,close,adj_close,volume
count,368251,368251.000000,368251.000000,368251.000000,368251.000000,349877.000000,3.682510e+05
mean,1998-04-19 05:03:19.204347,46.308766,46.771204,45.839768,46.316711,51.711536,3.095142e+07
min,1962-01-02 00:00:00,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,1985-05-28 00:00:00,4.210000,4.250000,4.170000,4.210000,1.395997,1.731400e+06
50%,2000-01-07 00:00:00,19.890000,20.090000,19.670000,19.900000,11.980000,4.649600e+06
75%,2012-08-21 00:00:00,57.560000,58.160000,56.990000,57.570000,43.790000,1.102151e+07
max,2024-12-06 00:00:00,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09
std,NaN,67.143015,67.789068,66.489203,67.154690,195.001654,1.266145e+08


After cleaning — numeric statistics:


,ts,open,high,low,close,adj_close,volume
count,368246,368246.000000,368246.000000,368246.000000,368246.000000,368246.000000,3.682460e+05
mean,1998-04-19 09:16:04.487869,46.309394,46.771824,45.840375,46.317325,37.274670,3.095184e+07
min,1962-01-02 00:00:00,0.005208,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,1985-05-28 00:00:00,4.210000,4.250000,4.170000,4.210000,1.310000,1.731514e+06
50%,2000-01-07 00:00:00,19.890000,20.090000,19.670000,19.910000,11.460000,4.649648e+06
75%,2012-08-21 00:00:00,57.560000,58.160000,56.997500,57.570000,41.570000,1.102160e+07
max,2024-12-06 00:00:00,604.260000,608.630000,597.880000,605.400000,605.400000,9.276606e+09
std,NaN,67.143254,67.789319,66.489450,67.154939,63.587872,1.266153e+08


In [25]:
# Artificial examples only; these do not modify the actual data.
demo_raw = pd.DataFrame({
    "ticker": ["AAA", "AAA", "AAA", "AAA", "BBB"],
    "ts": pd.to_datetime([
        "2024-01-02", "2024-01-03", "2024-01-04",
        "2024-01-05", "2024-01-02",
    ]),
    "open":   [10.0, np.nan, np.nan, np.nan, np.nan],
    "high":   [12.0, 12.0, np.nan, 25.0, 12.0],
    "low":    [9.0, 9.0, np.nan, 20.0, 9.0],
    "close":  [11.0, 11.0, np.nan, 22.0, 11.0],
    "volume": [100.0, np.nan, 100.0, 100.0, 100.0],
})

demo_candidates, demo_removed, demo_quarantine = clean_raw_data(demo_raw)

display(demo_candidates)
display(demo_removed)
display(demo_quarantine)

# AAA January 3: missing open fills from AAA January 2.
filled_row = demo_candidates.loc[
    demo_candidates["ts"].eq(pd.Timestamp("2024-01-03"))
].iloc[0]

assert filled_row["open"] == 10.0
assert filled_row["open_filled"]
assert filled_row["volume"] == 0
assert filled_row["volume_filled"]

# AAA January 4: all raw prices missing -> removed before filling.
assert len(demo_removed) == 1
assert demo_removed.iloc[0]["ts"] == pd.Timestamp("2024-01-04")

# AAA January 5: filled open lies outside today's range -> quarantine.
# BBB January 2: leading missing open cannot borrow AAA's value.
assert len(demo_quarantine) == 2
assert set(demo_quarantine["ticker"]) == {"AAA", "BBB"}
assert len(demo_candidates) == 2

print("Passed: filling, removal, ticker isolation, and bounds checks.")

,ticker,ts,open,high,low,close,volume,open_filled,high_filled,low_filled,close_filled,volume_filled,quarantine_reason
0,AAA,2024-01-02,10.0,12.0,9.0,11.0,100.0,False,False,False,False,False,
1,AAA,2024-01-03,10.0,12.0,9.0,11.0,0.0,True,False,False,False,True,


,ticker,ts,open,high,low,close,volume,removal_reason
2,AAA,2024-01-04,NaN,NaN,NaN,NaN,100.0,All four raw prices missing


,ticker,ts,open,high,low,close,volume,open_filled,high_filled,low_filled,close_filled,volume_filled,quarantine_reason
3,AAA,2024-01-05,10.0,25.0,20.0,22.0,100.0,True,False,False,False,False,Invalid raw OHLC bounds;
4,BBB,2024-01-02,NaN,12.0,9.0,11.0,100.0,False,False,False,False,False,"Missing, non-finite, or nonpositive raw price;..."


Passed: filling, removal, ticker isolation, and bounds checks.


In [26]:
demo_adjustments = pd.DataFrame({
    "ticker": ["TEST"] * 4,
    "ts": pd.date_range("2024-01-02", periods=4),
    "close": [100.0, 100.0, 100.0, 100.0],
    "adj_close": [np.nan, 999.0, 80.0, np.nan],
    "ref_close": [100.0, 100.0, 100.0, 100.0],
    "ref_adj_close": [50.0, 80.0, 80.0, np.nan],
})

demo_adjusted, demo_adj_quarantine = clean_adjusted_close(
    demo_adjustments
)

display(
    demo_adjusted[
        [
            "ts", "adj_close_original", "adjustment_factor_used",
            "adj_close", "adj_close_repaired",
        ]
    ]
)
display(demo_adj_quarantine)

assert np.allclose(
    demo_adjusted["adj_close"],
    [50.0, 80.0, 80.0],
)

assert demo_adjusted["adj_close_repaired"].tolist() == [
    True, True, False,
]

assert len(demo_adj_quarantine) == 1
assert demo_adj_quarantine.iloc[0]["ts"] == pd.Timestamp("2024-01-05")

print("Passed: date-specific repairs and unresolved-factor quarantine.")

,ts,adj_close_original,adjustment_factor_used,adj_close,adj_close_repaired
0,2024-01-02,NaN,0.5,50.0,True
1,2024-01-03,999.0,0.8,80.0,True
2,2024-01-04,80.0,0.8,80.0,False


,ticker,ts,close,adj_close,ref_close,ref_adj_close,adj_close_original,adj_close_repaired,adjustment_factor_used,adjustment_source,quarantine_reason
3,TEST,2024-01-05,100.0,NaN,100.0,NaN,NaN,False,NaN,ohlc: matched ticker and ts,Unresolved adjusted-close consistency


Passed: date-specific repairs and unresolved-factor quarantine.


In [27]:
demo_invalid = pd.DataFrame({
    "ticker": ["TEST"] * 3,
    "ts": pd.to_datetime([
        "2024-01-02", "2024-01-03", "2024-01-04",
    ]),
    "open": [10.0, 10.0, 10.0],
    "high": [12.0, 12.0, 12.0],
    "low": [9.0, 9.0, 9.0],
    "close": [13.0, np.inf, 11.0],
    "adj_close": [13.0, 11.0, -1.0],
    "volume": [100.0, 100.0, -5.0],
})

demo_invalid_issues = detect_price_volume_issues(demo_invalid)
display(demo_invalid_issues)

def has_issue(date, issue_type):
    """Check whether the example report contains a particular issue."""
    return (
        demo_invalid_issues["issue_date"].eq(pd.Timestamp(date))
        & demo_invalid_issues["issue_type"].eq(issue_type)
    ).any()

assert has_issue("2024-01-02", "invalid_ohlc_bounds")
assert has_issue("2024-01-03", "invalid_price")
assert has_issue("2024-01-04", "invalid_price")
assert has_issue("2024-01-04", "invalid_volume")

# Raw-price or volume failures should be quarantined.
demo_kept, demo_dropped, demo_bad = clean_raw_data(demo_invalid)

assert len(demo_kept) == 0
assert len(demo_dropped) == 0
assert len(demo_bad) == 3

print("Passed: invalid prices, OHLC bounds, and volume detected.")

,ticker,issue_type,issue_date,description,severity
0,TEST,invalid_price,2024-01-03,close is nonpositive or non-finite,high
1,TEST,invalid_price,2024-01-04,adj_close is nonpositive or non-finite,high
2,TEST,invalid_ohlc_bounds,2024-01-02,"close lies outside [low, high]",high
3,TEST,invalid_volume,2024-01-04,Volume is negative or non-finite,high


Passed: invalid prices, OHLC bounds, and volume detected.


In [28]:
# Mostly zero returns, with one large positive and one large negative move.
demo_returns = np.zeros(40)
demo_returns[10] = 0.50
demo_returns[30] = -0.50

tail_prices = 100 * np.cumprod(
    np.r_[1.0, 1.0 + demo_returns]
)
tail_dates = pd.date_range("2024-01-01", periods=len(tail_prices))

demo_return_data = pd.concat(
    [
        pd.DataFrame({
            "ticker": "TAILS",
            "ts": tail_dates,
            "close": tail_prices,
        }),
        pd.DataFrame({
            "ticker": "FLAT",
            "ts": pd.date_range("2024-01-01", periods=5),
            "close": 100.0,
        }),
        pd.DataFrame({
            "ticker": ["SHORT"],
            "ts": pd.to_datetime(["2024-01-01"]),
            "close": [100.0],
        }),
    ],
    ignore_index=True,
)

demo_return_issues = detect_return_outliers(demo_return_data)
display(demo_return_issues)

expected_dates = {tail_dates[11], tail_dates[31]}

assert len(demo_return_issues) == 2
assert set(demo_return_issues["ticker"]) == {"TAILS"}
assert set(demo_return_issues["issue_date"]) == expected_dates

print("Passed: both tails flagged; flat and insufficient histories handled.")

,ticker,issue_type,issue_date,description,severity
11,TAILS,extreme_return,2024-01-12,Raw-close simple return 50.00% exceeds three s...,medium
31,TAILS,extreme_return,2024-02-01,Raw-close simple return -50.00% exceeds three ...,medium


Passed: both tails flagged; flat and insufficient histories handled.


In [29]:
# Refresh return flags using the retained dataset.
after_return_keys = detect_return_outliers(cleaned)[
    ["ticker", "issue_date"]
].rename(columns={"issue_date": "ts"})

after_return_keys["extreme_return_flag_after"] = True

output = cleaned.merge(
    after_return_keys,
    on=["ticker", "ts"],
    how="left",
    validate="one_to_one",
)

output["extreme_return_flag_after"] = (
    output["extreme_return_flag_after"].eq(True)
)

audit_columns = [
    "ticker_original",
    "ticker_normalized",
    "open_filled",
    "high_filled",
    "low_filled",
    "close_filled",
    "volume_filled",
    "adj_close_original",
    "adj_close_repaired",
    "adjustment_factor_used",
    "adjustment_source",
    "extreme_return_flag",
    "extreme_return_flag_after",
    "zero_volume_flag",
]

output = output[data_columns + audit_columns].copy()

# SQL date filters need consistent ISO date strings.
output["ts"] = output["ts"].dt.strftime("%Y-%m-%d")

print("Rows ready to save:", len(output))
display(output.head())

Rows ready to save: 368246


,ts,open,high,low,close,adj_close,volume,ticker,ticker_original,ticker_normalized,...,low_filled,close_filled,volume_filled,adj_close_original,adj_close_repaired,adjustment_factor_used,adjustment_source,extreme_return_flag,extreme_return_flag_after,zero_volume_flag
0,1980-12-12,0.12835,0.12835,0.12835,0.12835,0.098941,469034068,AAPL,AAPL,False,...,False,False,False,0.098941,False,0.770866,ohlc: matched ticker and ts,False,False,False
1,1980-12-15,0.12165,0.12165,0.12165,0.12165,0.093779,175884971,AAPL,AAPL,False,...,False,False,False,0.093779,False,0.770891,ohlc: matched ticker and ts,False,False,False
2,1980-12-16,0.11272,0.11272,0.11272,0.11272,0.086896,105728103,AAPL,AAPL,False,...,False,False,False,0.086896,False,0.770899,ohlc: matched ticker and ts,False,False,False
3,1980-12-17,0.11551,0.11551,0.11551,0.11551,0.089046,86441684,AAPL,AAPL,False,...,False,False,False,0.089046,False,0.770898,ohlc: matched ticker and ts,False,False,False
4,1980-12-18,0.11886,0.11886,0.11886,0.11886,0.091627,73449671,AAPL,AAPL,False,...,False,False,False,0.091627,False,0.770885,ohlc: matched ticker and ts,False,False,False


In [30]:
assert db_path.is_file(), "Database file is missing"

with sqlite3.connect(db_path) as conn:
    raw_count_before = conn.execute(
        "SELECT COUNT(*) FROM ohlc_hw"
    ).fetchone()[0]

    output.to_sql(
        "ohlc_hw_cleaned",
        conn,
        if_exists="replace",
        index=False,
    )

    saved_count = conn.execute(
        "SELECT COUNT(*) FROM ohlc_hw_cleaned"
    ).fetchone()[0]

    raw_count_after = conn.execute(
        "SELECT COUNT(*) FROM ohlc_hw"
    ).fetchone()[0]

    saved_sample = pd.read_sql_query(
        """
        SELECT ticker, ts, close, adj_close, adj_close_repaired
        FROM ohlc_hw_cleaned
        ORDER BY ticker, ts
        LIMIT 5
        """,
        conn,
    )

assert saved_count == len(output)
assert raw_count_before == raw_count_after == len(df_raw)

print(f"Saved {saved_count:,} rows to ohlc_hw_cleaned.")
print("Original ohlc_hw row count is unchanged.")
display(saved_sample)

Saved 368,246 rows to ohlc_hw_cleaned.
Original ohlc_hw row count is unchanged.


,ticker,ts,close,adj_close,adj_close_repaired
0,AAPL,1980-12-12,0.12835,0.098941,0
1,AAPL,1980-12-15,0.12165,0.093779,0
2,AAPL,1980-12-16,0.11272,0.086896,0
3,AAPL,1980-12-17,0.11551,0.089046,0
4,AAPL,1980-12-18,0.11886,0.091627,0


In [31]:
audit_tables = {
    "hw2_quality_before": quality_report_before,
    "hw2_quality_after": quality_report_after,
    "hw2_quarantined": quarantined_rows,
    "hw2_removed_all_prices_missing": removed_rows,
}

with sqlite3.connect(db_path) as conn:
    for table_name, frame in audit_tables.items():
        saved_frame = frame.copy()

        for date_column in ["ts", "issue_date"]:
            if date_column in saved_frame.columns:
                saved_frame[date_column] = pd.to_datetime(
                    saved_frame[date_column]
                ).dt.strftime("%Y-%m-%d")

        saved_frame.to_sql(
            table_name,
            conn,
            if_exists="replace",
            index=False,
        )

        print(f"Saved {len(saved_frame):,} rows to {table_name}")

Saved 33,932 rows to hw2_quality_before
Saved 5,136 rows to hw2_quality_after
Saved 5 rows to hw2_quarantined
Saved 0 rows to hw2_removed_all_prices_missing
